In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
torchvision.set_image_backend('PIL')
from torchvision import transforms
from torchvision.utils import save_image
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

PyTorch Version: 2.10.0+cu128
CUDA Available: True


In [2]:

# configuration


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATASET_PATH = "/kaggle/input/datasets/shreshthsharmaa/celeba"
IMAGE_SIZE = 64
BATCH_SIZE = 128
EPOCHS = 50
LEARNING_RATE = 0.0002
BETA1 = 0.5
STEP_SIZE = 10
GAMMA = 0.5
LATENT_DIM = 100
FEATURES_GEN = 64
FEATURES_DISC = 64
CHANNELS_IMG = 3
CHECKPOINT_DIR = "checkpoints"
SAMPLE_DIR = "generated_samples"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLE_DIR, exist_ok=True)
print("Using Device:", DEVICE)

Using Device: cuda


In [3]:

# dataset & transforms

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.5 for _ in range(CHANNELS_IMG)],
        [0.5 for _ in range(CHANNELS_IMG)]
    )
])

dataset = torchvision.datasets.ImageFolder(
    root=DATASET_PATH,
    transform=transform
)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

print("Dataset Size:", len(dataset))


Dataset Size: 202599


In [4]:

# weights


def weights_init(model):
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.BatchNorm2d)):
            nn.init.normal_(m.weight.data, 0.0, 0.02)

In [5]:


# generator


class Generator(nn.Module):

    def __init__(self, z_dim, channels_img, features_g):
        super().__init__()

        self.net = nn.Sequential(
            self._block(z_dim, features_g * 16, 4, 1, 0),
            self._block(features_g * 16, features_g * 8, 4, 2, 1),
            self._block(features_g * 8, features_g * 4, 4, 2, 1),
            self._block(features_g * 4, features_g * 2, 4, 2, 1),

            nn.ConvTranspose2d(
                features_g * 2,
                channels_img,
                kernel_size=4,
                stride=2,
                padding=1
            ),

            nn.Tanh()
        )

    def _block(self, in_channels, out_channels, kernel_size, stride, padding):
        return nn.Sequential(
            nn.ConvTranspose2d(
                in_channels,
                out_channels,
                kernel_size,
                stride,
                padding,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(True)
        )

    def forward(self, x):
        return self.net(x)


In [6]:


# discriminator

class Discriminator(nn.Module):

    def __init__(self, channels_img, features_d):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(channels_img, features_d, 4, 2, 1),
            nn.LeakyReLU(0.2),

            self._block(features_d, features_d * 2, 4, 2, 1),
            self._block(features_d * 2, features_d * 4, 4, 2, 1),
            self._block(features_d * 4, features_d * 8, 4, 2, 1),

            nn.Conv2d(features_d * 8, 1, 4, 2, 0),
            
        )

    def _block(self, in_channels, out_channels, kernel_size, stride, padding):
        return nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size,
                stride,
                padding,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2)
        )

    def forward(self, x):
        return self.net(x)




In [7]:

# initialize models


generator = Generator(
    LATENT_DIM,
    CHANNELS_IMG,
    FEATURES_GEN
).to(DEVICE)

discriminator = Discriminator(
    CHANNELS_IMG,
    FEATURES_DISC
).to(DEVICE)

weights_init(generator)
weights_init(discriminator)

print("Models initialized successfully.")

Models initialized successfully.


In [8]:

# loss & optimizers


criterion = nn.BCEWithLogitsLoss()

optimizer_g = optim.Adam(
    generator.parameters(),
    lr=LEARNING_RATE,
    betas=(BETA1, 0.999)
)

optimizer_d = optim.Adam(
    discriminator.parameters(),
    lr=LEARNING_RATE,
    betas=(BETA1, 0.999)
)

scaler_g = GradScaler(device=DEVICE)
scaler_d = GradScaler(device=DEVICE)

In [9]:

# learning rate schedulers


scheduler_g = torch.optim.lr_scheduler.StepLR(
    optimizer_g,
    step_size=STEP_SIZE,
    gamma=GAMMA
)

scheduler_d = torch.optim.lr_scheduler.StepLR(
    optimizer_d,
    step_size=STEP_SIZE,
    gamma=GAMMA
)

print("Schedulers initialized.")

Schedulers initialized.


In [10]:

# checkpoint functions

def save_checkpoint(epoch):

    checkpoint = {
        "epoch": epoch,

        "generator_state_dict":
            generator.state_dict(),

        "discriminator_state_dict":
            discriminator.state_dict(),

        "optimizer_g_state_dict":
            optimizer_g.state_dict(),

        "optimizer_d_state_dict":
            optimizer_d.state_dict(),

        "scheduler_g_state_dict":
            scheduler_g.state_dict(),

        "scheduler_d_state_dict":
            scheduler_d.state_dict(),
    }

    checkpoint_path = os.path.join(
        CHECKPOINT_DIR,
        f"dcgan_checkpoint_epoch_{epoch}.pth"
    )

    torch.save(checkpoint, checkpoint_path)

    print(f"Checkpoint saved: {checkpoint_path}")


def load_checkpoint(checkpoint_path):

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE
    )

    generator.load_state_dict(
        checkpoint["generator_state_dict"]
    )

    discriminator.load_state_dict(
        checkpoint["discriminator_state_dict"]
    )

    optimizer_g.load_state_dict(
        checkpoint["optimizer_g_state_dict"]
    )

    optimizer_d.load_state_dict(
        checkpoint["optimizer_d_state_dict"]
    )

    scheduler_g.load_state_dict(
        checkpoint["scheduler_g_state_dict"]
    )

    scheduler_d.load_state_dict(
        checkpoint["scheduler_d_state_dict"]
    )

    start_epoch = checkpoint["epoch"] + 1

    print(f"Resumed from epoch {start_epoch}")

    return start_epoch




In [11]:


# save generated images

def save_generated_images(epoch):

    generator.eval()

    with torch.no_grad():

        noise = torch.randn(
            64,
            LATENT_DIM,
            1,
            1
        ).to(DEVICE)

        fake_images = generator(noise)

        save_image(
            fake_images,
            os.path.join(
                SAMPLE_DIR,
                f"epoch_{epoch}.png"
            ),
            normalize=True
        )

    generator.train()


In [12]:
# training loop

def train(
    start_epoch=1,
    total_epochs=EPOCHS,
    save_every=5
):

    generator.train()
    discriminator.train()

    for epoch in range(start_epoch, total_epochs + 1):

        for batch_idx, (real, _) in enumerate(dataloader):

            real = real.to(DEVICE)

            cur_batch_size = real.shape[0]

            noise = torch.randn(
                cur_batch_size,
                LATENT_DIM,
                1,
                1
            ).to(DEVICE)

           # training discriminator

            with autocast(device_type=DEVICE):

                fake = generator(noise)

                disc_real = discriminator(real).reshape(-1)

                loss_real = criterion(
                    disc_real,
                    torch.ones_like(disc_real)
                )

                disc_fake = discriminator(
                    fake.detach()
                ).reshape(-1)

                loss_fake = criterion(
                    disc_fake,
                    torch.zeros_like(disc_fake)
                )

                loss_d = (loss_real + loss_fake) / 2

            optimizer_d.zero_grad()

            scaler_d.scale(loss_d).backward()

            scaler_d.step(optimizer_d)

            scaler_d.update()

            #train generator

            with autocast(device_type=DEVICE):

                output = discriminator(fake).reshape(-1)

                loss_g = criterion(
                    output,
                    torch.ones_like(output)
                )

            optimizer_g.zero_grad()

            scaler_g.scale(loss_g).backward()

            scaler_g.step(optimizer_g)

            scaler_g.update()

            if batch_idx % 100 == 0:

                print(
                    f"Epoch [{epoch}/{total_epochs}] "
                    f"Batch [{batch_idx}/{len(dataloader)}] "
                    f"Loss D: {loss_d:.4f}, "
                    f"Loss G: {loss_g:.4f}"
                )

        save_generated_images(epoch)

        if epoch % save_every == 0:
            save_checkpoint(epoch)

        scheduler_g.step()
        scheduler_d.step()

        print(
            f"Generator LR: "
            f"{scheduler_g.get_last_lr()[0]}"
        )

        print(
            f"Discriminator LR: "
            f"{scheduler_d.get_last_lr()[0]}"
        )

    print("Training completed.")


In [13]:
train(
    start_epoch=1,
    total_epochs=EPOCHS,
    save_every=5
)

Epoch [1/50] Batch [0/1583] Loss D: 0.6913, Loss G: 0.7959
Epoch [1/50] Batch [100/1583] Loss D: 0.0754, Loss G: 3.2559
Epoch [1/50] Batch [200/1583] Loss D: 0.4053, Loss G: 2.4103
Epoch [1/50] Batch [300/1583] Loss D: 0.5668, Loss G: 1.5810
Epoch [1/50] Batch [400/1583] Loss D: 0.4147, Loss G: 1.9700
Epoch [1/50] Batch [500/1583] Loss D: 0.7893, Loss G: 1.6054
Epoch [1/50] Batch [600/1583] Loss D: 0.6144, Loss G: 1.4558
Epoch [1/50] Batch [700/1583] Loss D: 0.6099, Loss G: 1.2110
Epoch [1/50] Batch [800/1583] Loss D: 0.4413, Loss G: 1.7188
Epoch [1/50] Batch [900/1583] Loss D: 0.5359, Loss G: 2.4374
Epoch [1/50] Batch [1000/1583] Loss D: 0.4256, Loss G: 2.3771
Epoch [1/50] Batch [1100/1583] Loss D: 0.5900, Loss G: 2.4871
Epoch [1/50] Batch [1200/1583] Loss D: 0.3794, Loss G: 2.3709
Epoch [1/50] Batch [1300/1583] Loss D: 0.4592, Loss G: 1.5861
Epoch [1/50] Batch [1400/1583] Loss D: 0.4744, Loss G: 1.8822
Epoch [1/50] Batch [1500/1583] Loss D: 0.3859, Loss G: 2.0012
Generator LR: 0.0002

In [14]:
import shutil

# Path to your folder
folder_path = "/kaggle/working/generated_samples"

# Create zip archive
shutil.make_archive(
    "/kaggle/working/generated_samples",
    'zip',
    folder_path
)

print("Zip created!")

Zip created!


In [15]:
from IPython.display import FileLink

FileLink('/kaggle/working/generated_samples.zip')

/kaggle/working/generated_samples.zip